# EDA – Construction Site Worker PPE Detection (`ppe-5`)

Notebook phân tích khám phá dữ liệu (EDA) cho dataset YOLO/Roboflow phát hiện PPE của công nhân công trường.

> Chạy trên **Kaggle**: add dataset vào notebook (Add Input), notebook sẽ tự tìm `data.yaml` trong `/kaggle/input`. Bật Internet không cần thiết; GPU không cần.

**Cấu trúc thư mục kỳ vọng**
```
ppe-5/
└── data/
    ├── train/ (images/, labels/)
    ├── valid/ (images/, labels/)
    ├── test/  (images/, labels/)
    ├── README.dataset.txt
    ├── README.roboflow.txt
    └── data.yaml
```

**Nội dung**
1. Cấu hình & nạp dữ liệu
2. Thống kê tổng quan (split, class, số object/ảnh, ảnh không nhãn)
3. Phân tích bounding box (kích thước, tỉ lệ, vị trí)
4. Phân tích ảnh (kích thước, độ sáng, độ mờ)
5. Hiển thị ảnh mẫu kèm bbox
6. Kiểm tra chất lượng nhãn (bbox trùng, ngoài biên, label thiếu)
7. **Phát hiện & vẽ ảnh trùng nhau** (exact + near-duplicate + data leakage giữa các split)
8. Kết luận

## 1. Cấu hình & nạp dữ liệu

In [ ]:
import os, re, time, hashlib, warnings
import concurrent.futures as cf
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# ====== CHỈNH Ở ĐÂY ======
# Tự động tìm thư mục chứa data.yaml (Kaggle: /kaggle/input/...). Nếu tìm sai, gán tay:
# DATA_ROOT = Path("/kaggle/input/<ten-dataset>/ppe-5/data")
def find_data_root():
    for base in [Path("/kaggle/input"), Path("ppe-5"), Path(".")]:
        if base.exists():
            for y in sorted(base.rglob("data.yaml")):
                if any((y.parent / s).exists() for s in ["train", "valid", "test"]):
                    return y.parent
    raise FileNotFoundError("Không tìm thấy data.yaml - hãy gán DATA_ROOT thủ công")

DATA_ROOT = find_data_root()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")  # nơi ghi file kết quả
print("DATA_ROOT:", DATA_ROOT, "| OUT_DIR:", OUT_DIR)
SPLITS = ["train", "valid", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Ngưỡng Hamming (trên 64-bit pHash) coi là near-duplicate. 0 = giống hệt về hash, <=6 thường là gần như cùng ảnh
NEAR_DUP_THRESHOLD = 6
MAX_GROUPS_TO_PLOT = 12               # số nhóm trùng tối đa vẽ cho mỗi loại
MAX_IMGS_PER_GROUP = 6                # số ảnh tối đa vẽ trong 1 nhóm
SEED = 42
rng = np.random.default_rng(SEED)

# ====== SONG SONG CPU / GPU ======
cv2.setNumThreads(1)                                   # mỗi thread dùng 1 luồng OpenCV, tránh oversubscribe
N_WORKERS = min(32, (os.cpu_count() or 2) * 2)         # I/O + decode ảnh nhả GIL nên thread pool chạy song song thật
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, DEVICE = None, "cpu"
USE_GPU = DEVICE == "cuda"                              # đặt False để ép chạy CPU
print(f"CPU cores: {os.cpu_count()} | threads: {N_WORKERS} | so khớp pHash trên: "
      f"{torch.cuda.get_device_name(0) if USE_GPU else 'CPU (BLAS đa luồng)'}")

def pmap(fn, items, desc="", workers=N_WORKERS):
    """map song song bằng thread pool, giữ nguyên thứ tự, có progress bar."""
    items = list(items)
    with cf.ThreadPoolExecutor(workers) as ex:
        return list(tqdm(ex.map(fn, items), total=len(items), desc=desc))

class timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.perf_counter(); return self
    def __exit__(self, *a): print(f"⏱ {self.name}: {time.perf_counter() - self.t:.1f}s")

In [ ]:
# ---- data.yaml ----
with open(DATA_ROOT / "data.yaml") as f:
    cfg = yaml.safe_load(f)
print(yaml.dump(cfg, sort_keys=False, allow_unicode=True))

names = cfg["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
NUM_CLASSES = len(names)
print(f"\nSố class: {NUM_CLASSES}\n{list(enumerate(names))}")

cmap = plt.get_cmap("tab20")
CLASS_COLORS = {i: cmap(i % 20) for i in range(NUM_CLASSES)}

In [ ]:
def split_dirs(split):
    """Trả về (thư mục ảnh, thư mục label). Hỗ trợ cả split/images + split/labels lẫn split phẳng."""
    base = DATA_ROOT / split
    img_dir = base / "images" if (base / "images").exists() else base
    lbl_dir = base / "labels" if (base / "labels").exists() else base
    return img_dir, lbl_dir

rows = []
for split in SPLITS:
    img_dir, lbl_dir = split_dirs(split)
    if not img_dir.exists():
        print(f"[!] Không thấy split '{split}' tại {img_dir}")
        continue
    for p in sorted(img_dir.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            rows.append({"split": split, "img_path": str(p), "name": p.name,
                         "lbl_path": str(lbl_dir / (p.stem + ".txt"))})
df_img = pd.DataFrame(rows)
print(f"Tổng số ảnh: {len(df_img)}")
df_img.head()

In [ ]:
# ---- đọc thông tin ảnh SONG SONG: kích thước, độ sáng, độ mờ, MD5, pHash, dHash ----
# Mỗi file đọc đúng 1 lần từ đĩa: dùng chung bytes cho MD5 và decode (grayscale -> nhanh hơn đọc màu).
def _u64(bits):
    return np.uint64(int(np.packbits(bits.astype(np.uint8).ravel()).view(">u8")[0]))

def phash64(gray):
    g = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(g)[:8, :8].ravel()
    return _u64(d > np.median(d[1:]))

def dhash64(gray):
    g = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    return _u64(g[:, 1:] > g[:, :-1])

def process_image(p):
    with open(p, "rb") as f:
        raw = f.read()
    md5 = hashlib.md5(raw).hexdigest()
    gray = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return dict(w=0, h=0, brightness=np.nan, contrast=np.nan, blur=np.nan,
                    md5=md5, phash=np.uint64(0), dhash=np.uint64(0), corrupted=True)
    return dict(w=gray.shape[1], h=gray.shape[0],
                brightness=float(gray.mean()), contrast=float(gray.std()),
                blur=float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                md5=md5, phash=phash64(gray), dhash=dhash64(gray), corrupted=False)

with timer("Đọc + hash ảnh"):
    info = pmap(process_image, df_img["img_path"], desc="Đọc ảnh")
df_img = pd.concat([df_img, pd.DataFrame(info)], axis=1)
print("Ảnh lỗi/không đọc được:", int(df_img["corrupted"].sum()))
df_img.head()

In [ ]:
# ---- đọc annotation SONG SONG (YOLO: cls cx cy w h, hỗ trợ cả polygon -> lấy bbox bao) ----
def parse_label(r):
    lp = Path(r.lbl_path)
    if not lp.exists():
        return None
    out = []
    for line in lp.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        vals = list(map(float, parts[1:]))
        if len(vals) == 4:
            cx, cy, w, h = vals
        else:  # polygon
            xs, ys = np.array(vals[0::2]), np.array(vals[1::2])
            cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
            w, h = xs.max() - xs.min(), ys.max() - ys.min()
        out.append(dict(split=r.split, name=r.name, img_path=r.img_path, cls=cls, cx=cx, cy=cy, w=w, h=h))
    return out

with timer("Đọc nhãn"):
    parsed = pmap(parse_label, df_img.itertuples(), desc="Đọc nhãn")
missing_lbl = sum(x is None for x in parsed)
df_ann = pd.DataFrame([a for x in parsed if x for a in x])
df_ann["cls_name"] = df_ann["cls"].map(lambda i: names[i] if i < NUM_CLASSES else f"unk_{i}")
df_ann["area"] = df_ann["w"] * df_ann["h"]
df_ann["aspect"] = df_ann["w"] / df_ann["h"].replace(0, np.nan)
ann_by_img = {k: g for k, g in df_ann.groupby("img_path")}   # tra cứu nhanh khi vẽ ảnh

print(f"Tổng số bbox: {len(df_ann)} | ảnh thiếu file label: {missing_lbl}")
df_ann.head()

## 2. Thống kê tổng quan

In [ ]:
# số ảnh / split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cnt = df_img["split"].value_counts().reindex(SPLITS).dropna()
axes[0].bar(cnt.index, cnt.values, color=sns.color_palette("Set2"))
for i, v in enumerate(cnt.values):
    axes[0].text(i, v, f"{v}\n({v/cnt.sum():.1%})", ha="center", va="bottom")
axes[0].set_title("Số ảnh theo split")

cnt_b = df_ann["split"].value_counts().reindex(SPLITS).dropna()
axes[1].bar(cnt_b.index, cnt_b.values, color=sns.color_palette("Set2"))
for i, v in enumerate(cnt_b.values):
    axes[1].text(i, v, f"{v}\n({v/cnt_b.sum():.1%})", ha="center", va="bottom")
axes[1].set_title("Số bounding box theo split")
plt.tight_layout(); plt.show()

In [ ]:
# phân bố class (tổng + theo split)
class_cnt = df_ann["cls_name"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=class_cnt.values, y=class_cnt.index, ax=axes[0], palette="viridis")
for i, v in enumerate(class_cnt.values):
    axes[0].text(v, i, f" {v}", va="center")
axes[0].set_title("Số instance theo class (toàn bộ)")

ct = pd.crosstab(df_ann["cls_name"], df_ann["split"])[[s for s in SPLITS if s in df_ann["split"].unique()]]
(ct / ct.sum()).loc[class_cnt.index].plot(kind="barh", ax=axes[1])
axes[1].set_title("Tỉ lệ class trong từng split (kiểm tra phân bố đồng đều)")
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

imb = class_cnt.max() / class_cnt.min()
print(f"Imbalance ratio (max/min): {imb:.1f}x")
ct

In [ ]:
# số object / ảnh, ảnh không có nhãn
obj_per_img = df_ann.groupby("img_path").size()
df_img["n_obj"] = df_img["img_path"].map(obj_per_img).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_img["n_obj"], discrete=True, ax=axes[0])
axes[0].set_title("Số bbox / ảnh")
sns.boxplot(data=df_img, x="split", y="n_obj", order=[s for s in SPLITS if s in df_img.split.unique()], ax=axes[1])
axes[1].set_title("Số bbox / ảnh theo split")
plt.tight_layout(); plt.show()

print(df_img["n_obj"].describe())
print(f"\nẢnh không có bbox (background): {(df_img.n_obj == 0).sum()} ({(df_img.n_obj == 0).mean():.1%})")

In [ ]:
# co-occurrence giữa các class trong cùng 1 ảnh
pres = pd.crosstab(df_ann["img_path"], df_ann["cls_name"]).clip(upper=1)
co = pres.T.dot(pres)
plt.figure(figsize=(0.8 * NUM_CLASSES + 4, 0.7 * NUM_CLASSES + 3))
sns.heatmap(co, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Co-occurrence: số ảnh chứa đồng thời 2 class")
plt.show()

## 3. Phân tích bounding box

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(df_ann["w"], bins=50, ax=axes[0, 0]); axes[0, 0].set_title("Chiều rộng bbox (chuẩn hoá)")
sns.histplot(df_ann["h"], bins=50, ax=axes[0, 1]); axes[0, 1].set_title("Chiều cao bbox (chuẩn hoá)")
sns.histplot(np.log10(df_ann["area"].clip(lower=1e-6)), bins=50, ax=axes[1, 0])
axes[1, 0].set_title("log10(diện tích bbox / diện tích ảnh)")
sns.histplot(df_ann["aspect"].clip(upper=df_ann["aspect"].quantile(0.99)), bins=50, ax=axes[1, 1])
axes[1, 1].set_title("Aspect ratio (w/h)")
plt.tight_layout(); plt.show()

# Tỉ lệ small / medium / large (theo diện tích chuẩn hoá, tham khảo kiểu COCO)
bins = [0, 0.01, 0.09, 1.01]
df_ann["size_bucket"] = pd.cut(df_ann["area"], bins=bins, labels=["small (<1%)", "medium (1-9%)", "large (>9%)"])
print(pd.crosstab(df_ann["cls_name"], df_ann["size_bucket"], normalize="index").round(3))

In [ ]:
# kích thước bbox theo class
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = df_ann["cls_name"].value_counts().index
sns.boxplot(data=df_ann, y="cls_name", x="area", order=order, ax=axes[0], showfliers=False)
axes[0].set_title("Diện tích bbox theo class")
sns.boxplot(data=df_ann, y="cls_name", x="aspect", order=order, ax=axes[1], showfliers=False)
axes[1].set_title("Aspect ratio theo class")
plt.tight_layout(); plt.show()

In [ ]:
# heatmap vị trí tâm bbox
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
h = axes[0].hist2d(df_ann["cx"], df_ann["cy"], bins=40, range=[[0, 1], [0, 1]], cmap="magma")
axes[0].invert_yaxis(); axes[0].set_title("Heatmap tâm bbox (tất cả class)")
plt.colorbar(h[3], ax=axes[0])

sub = df_ann.sample(min(len(df_ann), 5000), random_state=SEED)
sns.scatterplot(data=sub, x="w", y="h", hue="cls_name", s=8, alpha=0.5, ax=axes[1])
axes[1].set_title("Scatter w vs h theo class"); axes[1].legend(bbox_to_anchor=(1, 1), fontsize=8)
plt.tight_layout(); plt.show()

## 4. Phân tích ảnh

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.scatterplot(data=df_img, x="w", y="h", hue="split", alpha=0.5, ax=axes[0, 0])
axes[0, 0].set_title("Kích thước ảnh (w x h)")
sns.histplot(data=df_img, x="brightness", hue="split", bins=40, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Độ sáng trung bình")
sns.histplot(data=df_img, x="contrast", hue="split", bins=40, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Độ tương phản (std của gray)")
sns.histplot(data=df_img, x=np.log10(df_img["blur"] + 1), hue="split", bins=40, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("log10(Laplacian variance) – càng thấp càng mờ")
plt.tight_layout(); plt.show()

print(df_img.groupby(["w", "h"]).size().sort_values(ascending=False).head(10).rename("count"))

In [ ]:
# Các ảnh tối nhất / sáng nhất / mờ nhất
def show_extremes(col, ascending, title, k=5):
    sub = df_img.dropna(subset=[col]).sort_values(col, ascending=ascending).head(k)
    fig, axes = plt.subplots(1, k, figsize=(3.4 * k, 3.4))
    for ax, (_, r) in zip(np.atleast_1d(axes), sub.iterrows()):
        ax.imshow(cv2.cvtColor(cv2.imread(r.img_path), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{col}={r[col]:.1f}", fontsize=9); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_extremes("brightness", True, "Ảnh tối nhất")
show_extremes("brightness", False, "Ảnh sáng nhất")
show_extremes("blur", True, "Ảnh mờ nhất")

## 5. Ảnh mẫu kèm bounding box

In [ ]:
def draw_boxes(ax, img_path, lw=2, show_label=True):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    ax.imshow(img)
    for r in ann_by_img.get(img_path, df_ann.iloc[0:0]).itertuples():
        x1, y1 = (r.cx - r.w / 2) * W, (r.cy - r.h / 2) * H
        ax.add_patch(patches.Rectangle((x1, y1), r.w * W, r.h * H, fill=False,
                                       edgecolor=CLASS_COLORS.get(r.cls, "red"), linewidth=lw))
        if show_label:
            ax.text(x1, max(y1 - 2, 0), r.cls_name, fontsize=7, color="white",
                    bbox=dict(facecolor=CLASS_COLORS.get(r.cls, "red"), alpha=0.8, pad=1, edgecolor="none"))
    ax.axis("off")

def show_samples(paths, ncols=4, title=None):
    n = len(paths); nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4 * nrows))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, p in zip(np.atleast_1d(axes).ravel(), paths):
        draw_boxes(ax, p)
    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

with_obj = df_img[df_img["n_obj"] > 0]
show_samples(with_obj.sample(min(8, len(with_obj)), random_state=SEED)["img_path"].tolist(), title="Ảnh ngẫu nhiên")

In [ ]:
# Mỗi class lấy vài ảnh mẫu
for cname in df_ann["cls_name"].value_counts().index:
    paths = df_ann[df_ann["cls_name"] == cname]["img_path"].drop_duplicates()
    show_samples(paths.sample(min(4, len(paths)), random_state=SEED).tolist(), title=f"Class: {cname}")

In [ ]:
# Ảnh dày đặc nhất (nhiều bbox) & vài ảnh không có nhãn
show_samples(df_img.sort_values("n_obj", ascending=False).head(4)["img_path"].tolist(), title="Ảnh có nhiều bbox nhất")
bg = df_img[df_img["n_obj"] == 0]
if len(bg):
    show_samples(bg.head(4)["img_path"].tolist(), title="Ảnh không có bbox (kiểm tra xem có thiếu nhãn không)")

## 6. Kiểm tra chất lượng nhãn

In [ ]:
issues = {}
issues["bbox ngoài biên [0,1]"] = int(((df_ann.cx - df_ann.w/2 < -1e-3) | (df_ann.cx + df_ann.w/2 > 1 + 1e-3) |
                                       (df_ann.cy - df_ann.h/2 < -1e-3) | (df_ann.cy + df_ann.h/2 > 1 + 1e-3)).sum())
issues["bbox w/h <= 0"] = int(((df_ann.w <= 0) | (df_ann.h <= 0)).sum())
issues["bbox rất nhỏ (area < 1e-4)"] = int((df_ann.area < 1e-4).sum())
issues["class id ngoài phạm vi"] = int((~df_ann.cls.between(0, NUM_CLASSES - 1)).sum())
issues["ảnh thiếu file label"] = missing_lbl
issues["ảnh không có bbox"] = int((df_img.n_obj == 0).sum())

# bbox trùng lặp chính xác trong cùng 1 ảnh
dup_box = df_ann.round(4).duplicated(subset=["img_path", "cls", "cx", "cy", "w", "h"], keep="first")
issues["bbox trùng y hệt trong cùng ảnh"] = int(dup_box.sum())

# bbox chồng lấn rất cao (IoU > 0.9) trong cùng 1 ảnh – VECTOR HOÁ bằng numpy (ma trận IoU n x n mỗi ảnh)
def high_iou_pairs(path, g, thr=0.9):
    a = g[["cx", "cy", "w", "h"]].values; cl = g["cls"].values
    if len(a) < 2:
        return []
    x1, y1, x2, y2 = a[:, 0] - a[:, 2]/2, a[:, 1] - a[:, 3]/2, a[:, 0] + a[:, 2]/2, a[:, 1] + a[:, 3]/2
    iw = np.clip(np.minimum(x2[:, None], x2[None]) - np.maximum(x1[:, None], x1[None]), 0, None)
    ih = np.clip(np.minimum(y2[:, None], y2[None]) - np.maximum(y1[:, None], y1[None]), 0, None)
    inter = iw * ih
    area = a[:, 2] * a[:, 3]
    iou = inter / (area[:, None] + area[None] - inter + 1e-12)
    ii, jj = np.where(np.triu(iou > thr, 1))
    return [(path, cl[i], cl[j], iou[i, j]) for i, j in zip(ii, jj)]

with timer("Kiểm tra IoU"):
    hi_iou = [p for path, g in ann_by_img.items() for p in high_iou_pairs(path, g)]
issues["cặp bbox IoU > 0.9 trong cùng ảnh"] = len(hi_iou)
pd.Series(issues, name="count").to_frame()

## 7. 🔁 Ảnh trùng nhau (Duplicate Detection)

Ba mức kiểm tra:
1. **Exact duplicate** – cùng MD5 (file giống hệt từng byte).
2. **Near-duplicate** – pHash gần nhau (Hamming ≤ `NEAR_DUP_THRESHOLD`): cùng ảnh nhưng khác resize/nén/flip nhẹ/augmentation nhẹ.
3. **Data leakage** – nhóm trùng xuất hiện ở **nhiều split khác nhau** (vd. train & valid/test) → điểm số đánh giá bị thổi phồng.

Ngoài ra, với Roboflow, tên file có dạng `<tên_gốc>_jpg.rf.<hash>.jpg`; các ảnh cùng `<tên_gốc>` là **bản augment của cùng 1 ảnh nguồn**.

In [ ]:
valid = df_img[~df_img["corrupted"]].reset_index(drop=True)

# ---------- Union-Find ----------
class UF:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def groups_from_uf(uf, n):
    d = defaultdict(list)
    for i in range(n): d[uf.find(i)].append(i)
    return [v for v in d.values() if len(v) > 1]

# ---------- 1) Exact duplicates (MD5) ----------
exact_groups = [g.index.tolist() for _, g in valid.groupby("md5") if len(g) > 1]
print(f"[Exact]  {len(exact_groups)} nhóm, {sum(len(g) for g in exact_groups)} ảnh liên quan")

# ---------- 2) Near duplicates (pHash, Hamming) – GPU / BLAS đa luồng ----------
# Mẹo: Hamming(a,b) = |a| + |b| - 2·(a·b) với a,b là vector bit {0,1}
#  => toàn bộ ma trận khoảng cách N x N chỉ là MỘT phép nhân ma trận (GPU: cực nhanh; CPU: BLAS đa luồng).
def hamming_pairs(hashes, thr, use_gpu=USE_GPU, budget=1e8):
    N = len(hashes)
    bits = np.unpackbits(hashes.astype(">u8").view(np.uint8).reshape(N, 8), axis=1).astype(np.float32)  # N x 64
    ch = max(64, int(budget // max(N, 1)))                    # số hàng mỗi chunk, giới hạn bộ nhớ
    out = {}
    if use_gpu:
        B = torch.from_numpy(bits).cuda(); S = B.sum(1)
    else:
        S = bits.sum(1)
    for s in tqdm(range(0, N, ch), desc="So khớp pHash (" + ("GPU" if use_gpu else "CPU") + ")"):
        if use_gpu:
            A = B[s:s + ch]
            d = (S[s:s + ch, None] + S[None, :] - 2 * (A @ B.T)).round()
            i, j = torch.nonzero(d <= thr, as_tuple=True)
            keep = (i + s) < j
            i, j = i[keep], j[keep]
            dv = d[i, j].cpu().numpy(); i = (i + s).cpu().numpy(); j = j.cpu().numpy()
        else:
            d = np.rint(S[s:s + ch, None] + S[None, :] - 2 * (bits[s:s + ch] @ bits.T))
            i, j = np.where(d <= thr)
            keep = (i + s) < j
            i, j = i[keep], j[keep]
            dv = d[i, j]; i = i + s
        out.update({(int(a), int(b)): int(v) for a, b, v in zip(i, j, dv)})
    return out

hashes = valid["phash"].values.astype(np.uint64)
N = len(hashes)
with timer("So khớp near-duplicate"):
    pair_dist = hamming_pairs(hashes, NEAR_DUP_THRESHOLD)
uf_near = UF(N)
for (a, b) in pair_dist: uf_near.union(a, b)
near_groups = groups_from_uf(uf_near, N)
print(f"[Near]   {len(near_groups)} nhóm, {sum(len(g) for g in near_groups)} ảnh liên quan (Hamming <= {NEAR_DUP_THRESHOLD})")

# ---------- 3) Roboflow source grouping ----------
def rf_source(name):
    m = re.match(r"^(.*?)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+", name, flags=re.I)
    return m.group(1) if m else None
valid["rf_src"] = valid["name"].map(rf_source)
rf_groups = [g.index.tolist() for k, g in valid.dropna(subset=["rf_src"]).groupby("rf_src") if len(g) > 1]
print(f"[Roboflow] {len(rf_groups)} nhóm ảnh cùng ảnh nguồn (augmentation)" if valid["rf_src"].notna().any()
      else "[Roboflow] Tên file không theo format Roboflow -> bỏ qua")

# ---------- Tổng hợp ----------
def summarize(groups, label):
    out = []
    for gi, g in enumerate(groups):
        splits = valid.loc[g, "split"].tolist()
        out.append(dict(type=label, group_id=gi, size=len(g), n_splits=len(set(splits)),
                        splits=dict(Counter(splits)), leakage=len(set(splits)) > 1))
    return pd.DataFrame(out)

df_dup = pd.concat([summarize(exact_groups, "exact"), summarize(near_groups, "near"),
                    summarize(rf_groups, "roboflow_src")], ignore_index=True)
df_dup.groupby("type").agg(groups=("group_id", "count"), images=("size", "sum"), leaking_groups=("leakage", "sum"))

In [ ]:
# Tỉ lệ ảnh trùng + leakage theo cặp split
dup_idx = set(i for g in near_groups for i in g)
valid["in_dup_group"] = valid.index.isin(dup_idx)
print("Tỉ lệ ảnh nằm trong nhóm near-duplicate theo split:")
print(valid.groupby("split")["in_dup_group"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "dup_imgs", "count": "total", "mean": "ratio"}).round(3))

# Với mỗi ảnh ở split A, có bản trùng ở split B không?
leak = defaultdict(int)
for g in near_groups:
    sp = valid.loc[g, "split"].tolist()
    for a in set(sp):
        for b in set(sp):
            if a != b: leak[(a, b)] += sp.count(a)
if leak:
    lk = pd.Series(leak).unstack(fill_value=0)
    plt.figure(figsize=(5, 4)); sns.heatmap(lk, annot=True, fmt="d", cmap="Reds")
    plt.title("Leakage: #ảnh ở split (hàng) có bản trùng ở split (cột)"); plt.show()
else:
    print("\n✅ Không phát hiện leakage giữa các split (theo ngưỡng hiện tại).")

In [ ]:
# ---------- Hàm vẽ nhóm ảnh trùng ----------
SPLIT_COLORS = {"train": "tab:blue", "valid": "tab:orange", "test": "tab:red"}

def plot_dup_group(idxs, title, show_boxes=True, dist_ref=None):
    idxs = idxs[:MAX_IMGS_PER_GROUP]
    n = len(idxs)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 4))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, idxs):
        r = valid.loc[i]
        if show_boxes:
            draw_boxes(ax, r.img_path, lw=1.5, show_label=False)
        else:
            ax.imshow(cv2.cvtColor(cv2.imread(r.img_path), cv2.COLOR_BGR2RGB)); ax.axis("off")
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(SPLIT_COLORS.get(r.split, "k")); sp.set_linewidth(4)
        short = r["name"] if len(r["name"]) <= 28 else r["name"][:12] + "…" + r["name"][-12:]
        t = f"[{r.split}] {short}\n{int(r.w)}x{int(r.h)} | {int(r.n_obj)} box"
        if dist_ref is not None and i != dist_ref:
            a, b = min(i, dist_ref), max(i, dist_ref)
            if (a, b) in pair_dist: t += f" | Hamming={pair_dist[(a, b)]}"
        ax.set_title(t, fontsize=8, color=SPLIT_COLORS.get(r.split, "k"))
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

def plot_groups(groups, label, only_leak=False, top=MAX_GROUPS_TO_PLOT):
    gs = list(groups)
    if only_leak:
        gs = [g for g in gs if valid.loc[g, "split"].nunique() > 1]
    gs = sorted(gs, key=len, reverse=True)[:top]
    if not gs:
        print(f"(Không có nhóm nào cho: {label})"); return
    print(f"=== {label}: vẽ {len(gs)} nhóm ===  (viền màu: xanh=train, cam=valid, đỏ=test)")
    for k, g in enumerate(gs):
        plot_dup_group(g, f"{label} #{k+1} – {len(g)} ảnh – splits: {dict(Counter(valid.loc[g,'split']))}", dist_ref=g[0])

### 7.1 Exact duplicates (cùng MD5)

In [ ]:
plot_groups(exact_groups, "EXACT duplicate")

### 7.2 Near-duplicates (pHash)

In [ ]:
plot_groups(near_groups, "NEAR duplicate")

### 7.3 ⚠️ Data leakage – nhóm trùng xuyên split (train ↔ valid ↔ test)

In [ ]:
plot_groups(near_groups, "LEAKAGE (trùng giữa các split)", only_leak=True)

### 7.4 Ảnh cùng nguồn Roboflow (augmentation)

In [ ]:
if rf_groups:
    print("Phân bố số bản augment trên mỗi ảnh nguồn:")
    print(pd.Series([len(g) for g in rf_groups]).value_counts().sort_index().rename("số nhóm"))
    plot_groups(rf_groups, "ROBOFLOW source group", top=6)

In [ ]:
# Annotation có nhất quán giữa các ảnh trùng không? (vd. cùng ảnh nhưng số bbox khác nhau -> nghi ngờ nhãn sai)
rows = []
for g in near_groups + exact_groups:
    sub = valid.loc[g]
    if sub["n_obj"].nunique() > 1:
        rows.append(dict(size=len(g), n_obj=sub["n_obj"].tolist(), splits=sub["split"].tolist(), names=sub["name"].tolist()))
incons = pd.DataFrame(rows)
print(f"Nhóm trùng nhưng số bbox KHÁC nhau: {len(incons)} / {len(near_groups) + len(exact_groups)}")
incons.head(10)

In [ ]:
# Vẽ các nhóm trùng có nhãn không nhất quán
bad = [g for g in near_groups if valid.loc[g, "n_obj"].nunique() > 1]
plot_groups(bad, "NHÃN KHÔNG NHẤT QUÁN giữa ảnh trùng", top=6)

In [ ]:
# Xuất danh sách ảnh trùng ra CSV để xử lý (xoá/gộp)
rows = []
for typ, groups in [("exact", exact_groups), ("near", near_groups)]:
    for gi, g in enumerate(groups):
        for i in g:
            r = valid.loc[i]
            rows.append(dict(type=typ, group_id=gi, split=r.split, name=r["name"], img_path=r.img_path, n_obj=r.n_obj))
pd.DataFrame(rows).to_csv(OUT_DIR / "duplicate_images.csv", index=False)
print(f"Đã lưu {OUT_DIR / 'duplicate_images.csv'} ({len(rows)} dòng)")

# Gợi ý: danh sách ảnh trong train nên loại bỏ vì trùng với valid/test
to_drop = []
for g in near_groups:
    sp = valid.loc[g, "split"]
    if sp.nunique() > 1 and (sp != "train").any():
        to_drop += valid.loc[g][valid.loc[g, "split"] == "train"]["img_path"].tolist()
print(f"Ảnh train nên cân nhắc loại bỏ (trùng với valid/test): {len(set(to_drop))}")

## 8. Tóm tắt

In [ ]:
n_leak = int(df_dup[(df_dup.type == "near")]["leakage"].sum()) if len(df_dup) else 0
print("=" * 60)
print(f"Tổng ảnh               : {len(df_img)}  ({dict(df_img.split.value_counts())})")
print(f"Tổng bbox              : {len(df_ann)}  | số class: {NUM_CLASSES}")
print(f"Imbalance (max/min)    : {imb:.1f}x")
print(f"Ảnh không nhãn         : {(df_img.n_obj == 0).sum()}")
print(f"Nhóm exact duplicate   : {len(exact_groups)}")
print(f"Nhóm near duplicate    : {len(near_groups)}  (trong đó leakage xuyên split: {n_leak})")
print("=" * 60)

### Gợi ý hành động tiếp theo
- **Trùng/leakage**: loại ảnh trùng khỏi `train` (hoặc split lại theo *nhóm* ảnh nguồn) để valid/test phản ánh đúng khả năng tổng quát hoá.
- **Mất cân bằng class**: cân nhắc oversampling, class-weight hoặc augmentation riêng cho class hiếm.
- **Object nhỏ**: nếu nhiều bbox `small`, tăng `imgsz` khi train hoặc dùng tiling/SAHI khi inference.
- **Nhãn nghi ngờ**: xem lại các ảnh không nhãn, bbox IoU cao, và nhóm trùng có nhãn khác nhau.